# Amazon Crawl Pipeline (10 categories)

Notebook này crawl dữ liệu Amazon bằng **Amazon Scraper API** trong `README.md`, sau đó tạo dataset có nhiều feature nhất có thể cho mỗi row product.

Pipeline:
1. Search sản phẩm theo 10 danh mục, nhiều page.
2. Lấy `asin` unique.
3. Enrich từng `asin` bằng `product-details` + `top-reviews`.
4. Flatten dữ liệu thành bảng analytics-ready.
5. Export `csv`, `parquet`, `jsonl`.

In [9]:
# If needed:
# !pip install requests pandas pyarrow

import json
import math
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

In [10]:
# ==============================
# 1) Config
# ==============================
BASE_URL = "https://amazon-scraper-api.omkar.cloud/amazon"
API_KEY = "ok_e0fe018cf2cb16fe65f03f4e4bcbb6a0"
COUNTRY_CODE = "US"

# 10 danh mục rộng hơn, query tổng quát để phủ dữ liệu đa dạng
CATEGORY_QUERIES = {
    "tech_laptops": "laptop",
    "tech_smartphones": "smartphone",
    "tech_tablets": "tablet",
    "home_kitchen": "home appliances",
    "beauty_personal_care": "beauty products",
    "sports_outdoors": "sports equipment",
    "books": "books",
    "baby": "baby products",
    "pet_supplies": "pet supplies",
    "fashion": "fashion",
}

PAGES_PER_CATEGORY = 120
SORT_BY = "best_sellers"
TARGET_ROWS = 10_000
MAX_PRODUCTS_PER_CATEGORY = None  # None = khong cat theo category
DEDUPE_BY_ASIN = False  # False de dat quy mo lon (giu theo seed category/query)

# Enrich sau search: de dat 10k trong quota, chi enrich 1 phan mau
ENRICH_MODE = "sample"  # "none" | "sample" | "full"
ENRICH_SAMPLE_SIZE = 800
MAX_TOP_REVIEWS_PER_PRODUCT = 3

REQUEST_TIMEOUT = 45
MIN_SLEEP = 0.1
MAX_SLEEP = 0.3
RETRY = 3

OUT_DIR = Path("data") / "amazon_crawl"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"API-Key": API_KEY}

print("Output folder:", OUT_DIR.resolve())

Output folder: C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\data\amazon_crawl


In [11]:
# ==============================
# 2) Helpers
# ==============================
def safe_json_dumps(v):
    try:
        return json.dumps(v, ensure_ascii=False)
    except Exception:
        return None


def request_json(endpoint, params, retries=RETRY, raise_on_final_error=True):
    url = f"{BASE_URL}/{endpoint}"
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT,
            )

            if resp.status_code == 200:
                return resp.json()

            # 401/403 thường là key sai hoặc không có quyền
            if resp.status_code in (401, 403):
                raise RuntimeError(
                    f"Auth error ({resp.status_code}) | endpoint={endpoint} | params={params}. Check API key."
                )

            # Retry nhóm lỗi transient từ phía server / rate limit
            if resp.status_code in (429, 500, 502, 503, 504):
                last_error = RuntimeError(
                    f"Transient HTTP {resp.status_code} | endpoint={endpoint} | params={params}"
                )
                if attempt < retries:
                    time.sleep(1.2 * attempt)
                    continue

            # Các status khác: raise luôn
            resp.raise_for_status()

        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.0 * attempt)
                continue

    # Hết retry
    if raise_on_final_error and last_error is not None:
        raise last_error

    return {}


def search_with_fallback(query, page, country_code, sort_primary=SORT_BY):
    """Gọi search với sort chính, nếu fail thì fallback về relevance để tránh vỡ pipeline."""
    # 1) primary sort
    primary_payload = request_json(
        endpoint="search",
        params={
            "query": query,
            "page": page,
            "country_code": country_code,
            "sort_by": sort_primary,
        },
        raise_on_final_error=False,
    )

    if isinstance(primary_payload, dict) and primary_payload.get("results") is not None:
        return primary_payload, sort_primary, None

    # 2) fallback sort
    fallback_sort = "relevance"
    fallback_payload = request_json(
        endpoint="search",
        params={
            "query": query,
            "page": page,
            "country_code": country_code,
            "sort_by": fallback_sort,
        },
        raise_on_final_error=False,
    )

    if isinstance(fallback_payload, dict) and fallback_payload.get("results") is not None:
        msg = f"Primary sort failed. Fallback success: {sort_primary} -> {fallback_sort}"
        return fallback_payload, fallback_sort, msg

    return {}, fallback_sort, f"Search failed on both sort_by={sort_primary} and {fallback_sort}"


def chunk_text(text, limit=1000):
    if text is None:
        return None
    s = str(text)
    return s if len(s) <= limit else s[:limit] + "..."


def extract_review_features(reviews):
    if not reviews:
        return {
            "top_reviews_count": 0,
            "avg_top_review_rating": None,
            "sum_top_review_helpful_votes": 0,
            "verified_purchase_ratio": None,
            "vine_review_ratio": None,
            "sample_review_titles": None,
            "sample_review_texts": None,
            "sample_review_dates": None,
            "sample_reviewer_names": None,
        }

    ratings = [r.get("rating") for r in reviews if isinstance(r.get("rating"), (int, float))]
    helpful_votes = [r.get("helpful_votes", 0) or 0 for r in reviews]

    verified = [1 if r.get("is_verified_purchase") else 0 for r in reviews]
    vine = [1 if r.get("is_vine_review") else 0 for r in reviews]

    return {
        "top_reviews_count": len(reviews),
        "avg_top_review_rating": round(sum(ratings) / len(ratings), 3) if ratings else None,
        "sum_top_review_helpful_votes": int(sum(helpful_votes)),
        "verified_purchase_ratio": round(sum(verified) / len(verified), 3) if verified else None,
        "vine_review_ratio": round(sum(vine) / len(vine), 3) if vine else None,
        "sample_review_titles": " | ".join([chunk_text(r.get("review_title"), 150) or "" for r in reviews[:3]]) or None,
        "sample_review_texts": " | ".join([chunk_text(r.get("review_text"), 300) or "" for r in reviews[:2]]) or None,
        "sample_review_dates": " | ".join([r.get("review_date") or "" for r in reviews[:3]]) or None,
        "sample_reviewer_names": " | ".join([r.get("reviewer_name") or "" for r in reviews[:3]]) or None,
    }


def flatten_product_row(category_name, query, search_item, details, top_reviews):
    # Base fields from search
    row = {
        "crawl_ts_utc": datetime.now(timezone.utc).isoformat(),
        "country_code": COUNTRY_CODE,
        "seed_category": category_name,
        "seed_query": query,
        "asin": search_item.get("asin"),
        "title": search_item.get("title"),
        "link": search_item.get("link"),
        "image_url": search_item.get("image_url"),
        "search_price": search_item.get("price"),
        "search_original_price": search_item.get("original_price"),
        "search_currency": search_item.get("currency"),
        "search_rating": search_item.get("rating"),
        "search_reviews": search_item.get("reviews"),
        "search_delivery_info": search_item.get("delivery_info"),
        "search_sales_volume": search_item.get("sales_volume"),
        "search_is_best_seller": search_item.get("is_best_seller"),
        "search_is_amazon_choice": search_item.get("is_amazon_choice"),
        "search_is_prime": search_item.get("is_prime"),
        "search_is_climate_friendly": search_item.get("is_climate_friendly"),
        "search_number_of_offers": search_item.get("number_of_offers"),
        "search_lowest_offer_price": search_item.get("lowest_offer_price"),
        "search_has_variations": search_item.get("has_variations"),
    }

    # Rich fields from product-details
    row.update({
        "product_name": details.get("product_name"),
        "slug": details.get("slug"),
        "parent_asin": details.get("parent_asin"),
        "landing_asin": details.get("landing_asin"),
        "brand_info": details.get("brand_info"),
        "brand_url": details.get("brand_url"),
        "current_price": details.get("current_price"),
        "original_price": details.get("original_price"),
        "unit_price": details.get("unit_price"),
        "unit_count": details.get("unit_count"),
        "currency": details.get("currency"),
        "availability": details.get("availability"),
        "condition": details.get("condition"),
        "number_of_offers": details.get("number_of_offers"),
        "delivery_info": details.get("delivery_info"),
        "estimated_delivery_date": details.get("estimated_delivery_date"),
        "rating": details.get("rating"),
        "reviews": details.get("reviews"),
        "is_bestseller": details.get("is_bestseller"),
        "is_amazon_choice": details.get("is_amazon_choice"),
        "is_prime": details.get("is_prime"),
        "is_climate_friendly": details.get("is_climate_friendly"),
        "sales_volume": details.get("sales_volume"),
        "main_image_url": details.get("main_image_url"),
        "has_video": details.get("has_video"),
        "video_thumbnail": details.get("video_thumbnail"),
        "has_aplus_content": details.get("has_aplus_content"),
        "has_brand_story": details.get("has_brand_story"),
        "main_category_id": (details.get("main_category") or {}).get("id"),
        "main_category_name": (details.get("main_category") or {}).get("name"),
        "key_features_count": len(details.get("key_features") or []),
        "additional_images_count": len(details.get("additional_image_urls") or []),
        "product_videos_count": len(details.get("product_videos") or []),
        "user_videos_count": len(details.get("user_videos") or []),
        "category_hierarchy_depth": len(details.get("category_hierarchy") or []),
        "variation_dimensions_count": len(details.get("variation_dimensions") or []),
        "all_variants_count": len(details.get("all_variants") or {}),
        "frequently_bought_together_count": len(details.get("frequently_bought_together") or []),
    })

    # rating distribution
    detailed_rating = details.get("detailed_rating") or {}
    row.update({
        "rating_dist_1": detailed_rating.get("1") if isinstance(detailed_rating, dict) else None,
        "rating_dist_2": detailed_rating.get("2") if isinstance(detailed_rating, dict) else None,
        "rating_dist_3": detailed_rating.get("3") if isinstance(detailed_rating, dict) else None,
        "rating_dist_4": detailed_rating.get("4") if isinstance(detailed_rating, dict) else None,
        "rating_dist_5": detailed_rating.get("5") if isinstance(detailed_rating, dict) else None,
    })

    # technical/product details as semi-structured columns
    tech = details.get("technical_details") or {}
    prod = details.get("product_details") or {}

    row.update({
        "technical_details_json": safe_json_dumps(tech),
        "product_details_json": safe_json_dumps(prod),
        "key_features_json": safe_json_dumps(details.get("key_features") or []),
        "full_description": chunk_text(details.get("full_description"), 2000),
        "category_hierarchy_json": safe_json_dumps(details.get("category_hierarchy") or []),
        "variation_dimensions_json": safe_json_dumps(details.get("variation_dimensions") or []),
        "variants_json": safe_json_dumps(details.get("variants") or {}),
        "all_variants_json": safe_json_dumps(details.get("all_variants") or {}),
        "additional_image_urls_json": safe_json_dumps(details.get("additional_image_urls") or []),
        "product_videos_json": safe_json_dumps(details.get("product_videos") or []),
        "top_reviews_json": safe_json_dumps(top_reviews),
    })

    # Review aggregate features
    row.update(extract_review_features(top_reviews))

    # Numeric engineered features
    cp = row.get("current_price")
    op = row.get("original_price")
    if isinstance(cp, (int, float)) and isinstance(op, (int, float)) and op > 0:
        row["discount_pct"] = round((op - cp) / op * 100, 3)
    else:
        row["discount_pct"] = None

    row["title_len"] = len(row["title"]) if row.get("title") else None
    row["product_name_len"] = len(row["product_name"]) if row.get("product_name") else None

    return row

In [ ]:
# ==============================
# 3) Crawl: search -> details -> reviews
# ==============================
all_search_items = []
search_errors = []

for category_name, query in CATEGORY_QUERIES.items():
    print(f"\n[SEARCH] category={category_name} | query={query}")
    category_items = []

    for page in range(1, PAGES_PER_CATEGORY + 1):
        payload, used_sort, warn_msg = search_with_fallback(
            query=query,
            page=page,
            country_code=COUNTRY_CODE,
            sort_primary=SORT_BY,
        )

        page_items = payload.get("results", []) if isinstance(payload, dict) else []
        category_items.extend(page_items)

        if warn_msg:
            print(f"  - page={page} -> warning: {warn_msg}")
            search_errors.append(
                {
                    "stage": "search",
                    "seed_category": category_name,
                    "seed_query": query,
                    "page": page,
                    "error": warn_msg,
                }
            )

        print(f"  - page={page} | sort={used_sort} -> {len(page_items)} items")

        # Neu page khong con ket qua thi bo qua category hien tai
        if len(page_items) == 0:
            print(f"  - page={page} -> 0 items, skip remaining pages of this category")
            break

        time.sleep(random.uniform(MIN_SLEEP, MAX_SLEEP))

        # stop sớm nếu đã đạt target rows thô
        if len(all_search_items) + len(category_items) >= TARGET_ROWS:
            break

    if isinstance(MAX_PRODUCTS_PER_CATEGORY, int):
        category_items = category_items[:MAX_PRODUCTS_PER_CATEGORY]

    # attach seed metadata
    for item in category_items:
        item["_seed_category"] = category_name
        item["_seed_query"] = query

    all_search_items.extend(category_items)

    if len(all_search_items) >= TARGET_ROWS:
        break

# cắt đúng target
all_search_items = all_search_items[:TARGET_ROWS]
print(f"\nTotal raw search items (capped): {len(all_search_items)}")

if DEDUPE_BY_ASIN:
    asin_to_search = {}
    for item in all_search_items:
        asin = item.get("asin")
        if asin and asin not in asin_to_search:
            asin_to_search[asin] = item
    crawl_items = list(asin_to_search.values())
    print(f"Items after dedupe by ASIN: {len(crawl_items)}")
else:
    crawl_items = all_search_items
    print(f"Items without dedupe: {len(crawl_items)}")

rows = []
errors = []

# chọn subset để enrich khi cần scale lớn
if ENRICH_MODE == "full":
    enrich_indices = set(range(len(crawl_items)))
elif ENRICH_MODE == "sample":
    n = min(ENRICH_SAMPLE_SIZE, len(crawl_items))
    enrich_indices = set(random.sample(range(len(crawl_items)), n)) if n > 0 else set()
else:
    enrich_indices = set()

print(f"Enrich mode={ENRICH_MODE} | enrich rows={len(enrich_indices)}")

for idx, item in enumerate(crawl_items, start=1):
    asin = item.get("asin")
    category_name = item.get("_seed_category")
    query = item.get("_seed_query")

    try:
        details = {}
        reviews = []

        if (idx - 1) in enrich_indices and asin:
            details = request_json(
                endpoint="product-details",
                params={"asin": asin, "country_code": COUNTRY_CODE},
                raise_on_final_error=False,
            )

            reviews_payload = request_json(
                endpoint="product-reviews/top",
                params={"asin": asin, "country_code": COUNTRY_CODE},
                raise_on_final_error=False,
            )
            reviews = (reviews_payload or {}).get("results", [])[:MAX_TOP_REVIEWS_PER_PRODUCT]

        row = flatten_product_row(category_name, query, item, details or {}, reviews)
        row["is_enriched"] = (idx - 1) in enrich_indices
        rows.append(row)

        if idx % 200 == 0:
            print(f"Processed {idx}/{len(crawl_items)} items")

        time.sleep(random.uniform(MIN_SLEEP, MAX_SLEEP))

    except Exception as e:
        errors.append({"asin": asin, "error": str(e), "seed_category": category_name, "seed_query": query})

# Gộp lỗi search vào bảng lỗi chung
if search_errors:
    errors.extend(search_errors)

print(f"\nDone. Success rows: {len(rows)} | Errors: {len(errors)}")


[SEARCH] category=tech_laptops | query=laptop
  - page=1 | sort=best_sellers -> 16 items
  - page=2 | sort=best_sellers -> 16 items
  - page=3 | sort=best_sellers -> 16 items
  - page=4 | sort=best_sellers -> 16 items
  - page=5 | sort=best_sellers -> 16 items
  - page=6 | sort=best_sellers -> 16 items
  - page=7 | sort=best_sellers -> 16 items
  - page=8 | sort=best_sellers -> 16 items
  - page=9 | sort=best_sellers -> 16 items
  - page=10 | sort=best_sellers -> 16 items
  - page=11 | sort=best_sellers -> 16 items
  - page=12 | sort=best_sellers -> 16 items
  - page=13 | sort=best_sellers -> 16 items
  - page=14 | sort=best_sellers -> 16 items
  - page=15 | sort=best_sellers -> 16 items
  - page=16 | sort=best_sellers -> 16 items
  - page=17 | sort=best_sellers -> 16 items
  - page=18 | sort=best_sellers -> 16 items
  - page=19 | sort=best_sellers -> 16 items
  - page=20 | sort=best_sellers -> 2 items
  - page=21 | sort=best_sellers -> 0 items
  - page=22 | sort=best_sellers -> 0 ite

In [ ]:
# ==============================
# 4) Build dataframe + quality checks
# ==============================
df = pd.DataFrame(rows)
err_df = pd.DataFrame(errors)

print("df shape:", df.shape)
print("error shape:", err_df.shape)

if not df.empty:
    # Optional: prioritize important columns for display
    first_cols = [
        "crawl_ts_utc", "country_code", "seed_category", "seed_query", "asin",
        "title", "product_name", "brand_info", "current_price", "original_price",
        "discount_pct", "currency", "rating", "reviews", "availability",
        "is_prime", "is_bestseller", "is_amazon_choice", "sales_volume",
        "key_features_count", "all_variants_count", "top_reviews_count",
        "avg_top_review_rating", "verified_purchase_ratio"
    ]
    ordered_cols = [c for c in first_cols if c in df.columns] + [c for c in df.columns if c not in first_cols]
    df = df[ordered_cols]

    print("\nNull rate by selected columns:")
    check_cols = [c for c in ["asin", "title", "current_price", "rating", "reviews", "technical_details_json", "top_reviews_json"] if c in df.columns]
    print((df[check_cols].isna().mean() * 100).round(2))

df.head(3)

In [ ]:
# ==============================
# 5) Export outputs
# ==============================
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.csv"
parquet_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.parquet"
jsonl_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.jsonl"

err_csv_path = OUT_DIR / f"amazon_errors_{COUNTRY_CODE}_{ts}.csv"

if not df.empty:
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_parquet(parquet_path, index=False)
    df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

if not err_df.empty:
    err_df.to_csv(err_csv_path, index=False, encoding="utf-8-sig")

print("Saved:")
if not df.empty:
    print("-", csv_path.resolve())
    print("-", parquet_path.resolve())
    print("-", jsonl_path.resolve())
if not err_df.empty:
    print("-", err_csv_path.resolve())

In [ ]:
# ==============================
# 6) Optional: quick profiling
# ==============================
if not df.empty:
    numeric_cols = [c for c in ["current_price", "original_price", "discount_pct", "rating", "reviews", "top_reviews_count", "avg_top_review_rating"] if c in df.columns]
    display(df[numeric_cols].describe(include="all").T)

    print("\nRows per seed category:")
    display(df["seed_category"].value_counts())

    print("\nSample columns count:", len(df.columns))